# Run Model 1 and Model 2
The website uses the same shared functions as this notebook. Training code is not imported to run the models.
Supply a trained Model 1 checkpoint, its tokenizer folder, and a Model 2 adapter with `run.json`.
The base model files must be available locally. The default deployment uses a compatible CUDA GPU.
This notebook does not train or modify either model.

## 1. Set the artifact paths
Use the detection threshold accepted for this checkpoint on validation data. Do not choose it from test results.

In [ ]:
from pathlib import Path
import json
from models.service import load_service, prediction_failure, checked_generation
from models.model2 import prompt

PRIVATE_ROOT = Path.home() / "patternlab-private"
CHECKPOINT = PRIVATE_ROOT / "runs" / "model1-presentation" / "best.pt"
ADAPTER = PRIVATE_ROOT / "runs" / "model2-presentation"
DEVICE = "cuda"
THRESHOLD = 0.5  # Replace this with this checkpoint's accepted validation threshold.
TEXT = "Yesterday I polish the lantern."
assert CHECKPOINT.is_file() and (ADAPTER / "run.json").is_file(), "Set the trained model paths first."

## 2. Load both models once
The loader checks category order, checkpoint weights, the base model revision, and the prompt hash.
Model 1 stays in evaluation mode. Model 2 uses the trained adapter and deterministic greedy decoding.

In [ ]:
service = load_service(CHECKPOINT, ADAPTER, device=DEVICE, threshold=THRESHOLD,
                       local_files_only=True)

## 3. Run Model 1
The result contains an error probability, a category, and character positions. A negative result does not
prove that the whole sentence is correct. The system covers seven target categories and one error at a time.

In [ ]:
prediction = service.predictor.predict(TEXT)
print(json.dumps(prediction, indent=2, ensure_ascii=False))

## 4. Check the diagnosis and inspect the prompt
Only a valid positive prediction goes to Model 2. A replacement span covers existing text.
An insertion has equal start and end positions. The prompt must use exactly the span from Model 1.

In [ ]:
failure = prediction_failure(TEXT, prediction)
valid_positive = failure is None and prediction["has_error"]
if valid_positive:
    valid_positive = service.predictor.aligned(TEXT, prediction["span"])
if valid_positive:
    print(prompt(TEXT, prediction["category"], prediction["span"]))
else:
    print({"reason": failure or "No aligned target error. Model 2 is skipped."})

## 5. Generate and validate one explanation and edit
The validator rejects malformed JSON, changed spans, and invalid edits. It builds the corrected sentence
from the original text and one replacement. A failed validation produces no correction.

In [ ]:
if valid_positive:
    raw = service.generator.generate(TEXT, prediction["category"], prediction["span"])
    checked = checked_generation(TEXT, prediction["span"], raw)
    if checked is None:
        print({"status": "generation_failed", "reason": "invalid_generation"})
    else:
        generation, corrected_text = checked
        print(json.dumps({"generation": generation, "corrected_text": corrected_text},
                         indent=2, ensure_ascii=False))

## 6. Use the complete service as the website does
`diagnose_with_prediction()` runs the full pipeline with the same checks and failure statuses.
This cell repeats inference once so you can compare the service response with the steps above.
The website displays Chinese by default. Reviewer mode also displays English.

In [ ]:
result = service.diagnose_with_prediction(TEXT)
print(json.dumps(result, indent=2, ensure_ascii=False))

## 7. Start the website in a terminal
Keep the notebooks for the code walkthrough. Run the website in a separate process.
Use `python -m web --sample` for clearly labeled prewritten examples without weights.
For live inference, use the checkpoint, adapter, device, and threshold flags in the README.
Lessons, practice, quizzes, progress, copying, and sentence selection keep their existing behavior.